In [1]:
import requests
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timezone


Create tables

In [2]:
conn = sqlite3.connect("screener.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pools (
    pair_address TEXT PRIMARY KEY,
    token_address TEXT,
    chain_id TEXT,
    symbol TEXT,
    dex_id TEXT,
    price_usd REAL,
    liquidity_usd REAL,
    volume_m5 REAL,
    volume_h1 REAL,
    volume_h24 REAL,
    price_change_m5 REAL,
    price_change_h1 REAL,
    price_change_h24 REAL,
    market_cap REAL,
    fdv REAL,
    pair_created_at INTEGER,
    first_seen_at TIMESTAMP,
    last_updated_at TIMESTAMP
)
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pool_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        pair_address     TEXT REFERENCES pools(pair_address),
        price_usd        REAL,
        liquidity_usd    REAL,
        volume_m5        REAL,
        volume_h1        REAL,
        volume_h24       REAL,
        price_change_m5  REAL,
        price_change_h1  REAL,
        price_change_h24 REAL,
        market_cap       REAL,
        fdv              REAL,
        snapshot_at      TIMESTAMP
    )
""")

In [3]:
conn.commit()

response = requests.get("https://api.dexscreener.com/token-profiles/latest/v1", headers={"Accept": "*/*"})
tokens = response.json()
all_pairs = []
new_tokens_count = 0

for token in tokens:
    chain_id = token.get("chainId")
    token_address = token.get("tokenAddress")

    cursor.execute(
        """SELECT 1 FROM pools WHERE token_address = ? """,
        (token_address,)
    )

    if cursor.fetchone():
        continue

    print(f"New token discovered: {token_address}")
    new_tokens_count += 1

    now = datetime.now(timezone.utc).isoformat()
    data = requests.get(f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}",headers={"Accept": "*/*"})
    pair_response = data.json()

    pair_details = [{ "token_address" : token_address, 
                     "symbol" : item.get('baseToken').get('symbol'),
                     "pair_address" : item.get('pairAddress'),
                     "dex_id" : item.get('dexId'),
                     "price_usd" : item.get('priceUsd'),
                     "price_change_m5" : item.get('priceChange', {}).get('m5'),
                     "price_change_h1" : item.get('priceChange', {}).get('h1'),
                     "price_change_h24" : item.get('priceChange', {}).get('h24'),
                     "liquidity_usd" : item.get('liquidity', {}).get('usd'),
                     "volume_m5" : item.get('volume', {}).get('m5'),
                     "volume_h1" : item.get('volume', {}).get('h1'),
                     "volume_h24" : item.get('volume', {}).get('h24'),
                     "market_cap" : item.get('marketCap'),
                     "fdv" : item.get('fdv'),
                     "pair_created_at" : item.get('pairCreatedAt'),
                     "timestamp_fetched" : now,
    }
    for item in pair_response ]
    all_pairs.extend(pair_details)

    for pool in pair_details:
        cursor.execute(
            """
            INSERT INTO pools (
                pair_address,
                token_address,
                chain_id,
                symbol,
                dex_id,
                price_usd,
                liquidity_usd,
                volume_m5,
                volume_h1,
                volume_h24,
                price_change_m5,
                price_change_h1,
                price_change_h24,
                market_cap,
                fdv,
                pair_created_at,
                first_seen_at,
                last_updated_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) ON CONFLICT(pair_address) DO UPDATE SET
                price_usd       = excluded.price_usd,
                liquidity_usd   = excluded.liquidity_usd,
                volume_m5       = excluded.volume_m5,
                volume_h1       = excluded.volume_h1,
                volume_h24      = excluded.volume_h24,
                price_change_m5  = excluded.price_change_m5,
                price_change_h1  = excluded.price_change_h1,
                price_change_h24 = excluded.price_change_h24,
                market_cap      = excluded.market_cap,
                fdv             = excluded.fdv,
                last_updated_at = excluded.last_updated_at
            """,
            (
                pool["pair_address"],
                pool["token_address"],
                chain_id,
                pool["symbol"],
                pool["dex_id"],
                pool["price_usd"],
                pool["liquidity_usd"],
                pool["volume_m5"],
                pool["volume_h1"],
                pool["volume_h24"],
                pool["price_change_m5"],
                pool["price_change_h1"],
                pool["price_change_h24"],
                pool["market_cap"],
                pool["fdv"],
                pool["pair_created_at"],
                now,
                now
            )
        )
        cursor.execute(
            """
            INSERT INTO pool_snapshots (
                pair_address, 
                price_usd, 
                liquidity_usd,
                volume_m5, 
                volume_h1, 
                volume_h24,
                price_change_m5, 
                price_change_h1, 
                price_change_h24,
                market_cap, 
                fdv, snapshot_at
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
            """,
            (
                pool["pair_address"], 
                pool["price_usd"], 
                pool["liquidity_usd"],
                pool["volume_m5"], 
                pool["volume_h1"], 
                pool["volume_h24"],
                pool["price_change_m5"], 
                pool["price_change_h1"], 
                pool["price_change_h24"],
                pool["market_cap"], 
                pool["fdv"], now
            )
)      
conn.commit()

df = pd.DataFrame(all_pairs)
print(f"New tokens discovered: {new_tokens_count}")
print(f"Pairs collected: {len(df)}")
conn.close()

New token discovered: 83KSngkonyyJCuEQ7vfBnCFKyfGk6FttQnuQToUppump
New token discovered: 5vnDrcjuGZhbAaFocz9rYUsCrxg2gTz93Rrpghgtpump
New token discovered: 61qNX1UoKDMypgDJzNGFra3Mb6DQPyUmf6zDayZ7bonk
New token discovered: 0x82790bc500d677101986a696d341a156
New token discovered: BLSuVcXQ1hMMwFjnxw2KKaXYTcBCSMYt7pNLhtmqpump
New token discovered: GX2cScWJVWWatXTsaHpAfqKa8EzpdQ3hxBtmDs6Epump
New token discovered: HyiW1S4x4UYB7B1FNMMS8p3VcKbNyDrA3RqF7XP7pump
New token discovered: rbEEjCfCEWRMwtDoykokcA5HJEmvqvTXdLuKmvSpump
New token discovered: 8mrSTMNRnv8PLdM2ccGvA4uLmQpKbBHPfhsFRtVQpump
New token discovered: 6CatsKuvnwP4Vnucevkzg5itdxS9GGUXjNYrgEJJpump
New token discovered: C3GFrWEHJaTyTVjuauPsC5dB8JFG923bVjDEFoAEpump
New token discovered: BuiRWf6YrvDcS7neHPWvZGWzeivdRRwWZwhJBabhpump
New token discovered: u3h4mj8PRhdeS1GNmTStod95Bi223ANKAYtcrPLpump
New token discovered: 8knwhjZp5751BYFYupn1dXSaf8WMErVroqtvRUxnpump
New token discovered: MiX8b2uYqaZESCejL91W8zUMBMpfxQm6jk3mQRzpump
New toke

In [4]:
%reload_ext sql
%sql sqlite:////Users/ismail/GitHub_Projects/Solana-dexscreener_bot/screener.db

Connecting to 'sqlite:////Users/ismail/GitHub_Projects/Solana-dexscreener_bot/screener.db'

In [5]:
%%sql
SELECT pair_address,
       COUNT(*)
FROM pool_snapshots
GROUP BY pair_address
ORDER BY COUNT(*) DESC;

Running query in 'sqlite:////Users/ismail/GitHub_Projects/Solana-dexscreener_bot/screener.db'

pair_address,COUNT(*)
svoeQQ5UNdqyp9JsLff2WW4d1Zs8CrUweREmRgCFxwB,1
g5ufXD4f7qfLadJmyGrtjomUSyTxt52fFNGtyCREdgq,1
QmwUETTpSfoW26heJzNFUWuZV9bfxkyWD2N1SmepfkS,1
NmqSSRyoewmyXhAP1oZtp1EHuNd4evSxWPPEttRxGKn,1
Lh9nCQW2pmfegjcXmRJ1ACMgMHNv7NkSZi9NZbdUBXr,1
JCjHr1DU83VendcisP3y8pd2SwZ5bcnrFWWZqumxZ96V,1
HwcqTdc4c7x2U14GhPasWJoKhswSynj76ngU8FnxFdKM,1
HqeUNGhho9oRssK73NSKDHYouqqVLsx6kXjMbhHRRRwf,1
HpREJWu7fpqaWUmzEC9vK9o4Ek57dwUoMP1GESZEHH7h,1
HnpxcXSDoq615Lgwb5NPLTtZSgjZrwJ4EcdEYBcvvzXh,1
